In [1]:
!pip -q install transformers accelerate sentencepiece evaluate sacrebleu bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.7 MB/s eta 0:00:00


In [10]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
import re

from transformers import pipeline
import evaluate
from sacrebleu.metrics import BLEU

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 1. Load CSVs
intel_path = "drive/My Drive/cs263_final_proj/intel_misclassified_top200.csv"
pubhealth_path = "drive/My Drive/cs263_final_proj/pubhealth_misclassified_top200.csv"

intel_df = pd.read_csv(intel_path)
pub_df = pd.read_csv(pubhealth_path)

print("Intel cols:", list(intel_df.columns))
print("PubHealth cols:", list(pub_df.columns))

display(intel_df.head(2))
display(pub_df.head(2))

Intel cols: ['text', 'reasoning', 'label', 'model', 'label_text', 'labels', 'idx', 'true_id', 'true_label', 'pred_id', 'pred_label', 'pred_confidence', 'prob_true', 'prob_false', 'prob_mixture', 'model_input_text']
PubHealth cols: ['claim_id', 'claim', 'date_published', 'explanation', 'fact_checkers', 'main_text', 'sources', 'labels', 'subjects', 'idx', 'true_id', 'true_label', 'pred_id', 'pred_label', 'pred_confidence', 'prob_true', 'prob_false', 'prob_mixture', 'model_input_text']


,text,reasoning,label,model,label_text,labels,idx,true_id,true_label,pred_id,pred_label,pred_confidence,prob_true,prob_false,prob_mixture,model_input_text
0,Studies have consistently shown that social me...,This is false because numerous studies have de...,0,meta-llama/Meta-Llama-3.1-8B-Instruct,false,1,3200,1,false,0,true,0.998243,0.998243,0.000421,0.001336,[CLAIM] Studies have consistently shown that s...
1,AI systems will eventually surpass human intel...,This statement contains some truth because AI ...,2,meta-llama/Meta-Llama-3.1-8B-Instruct,partially_true,2,748,2,mixture,1,false,0.997742,0.000470,0.997742,0.001788,[CLAIM] AI systems will eventually surpass hum...


,claim_id,claim,date_published,explanation,fact_checkers,main_text,sources,labels,subjects,idx,true_id,true_label,pred_id,pred_label,pred_confidence,prob_true,prob_false,prob_mixture,model_input_text
0,9975,Crohn’s patients respond to J&J’s Stelara in s...,"May 8, 2011","Without evaluating the evidence, it becomes on...",NaN,"While we’re given the size of the market, we d...",NaN,2,Reuters Health,1165,2,mixture,0,true,0.997357,0.997357,0.000357,0.002286,[CLAIM] Crohn’s patients respond to J&J’s Stel...
1,30893,A teenage schoolgirl in Texas became pregnant ...,"October 5, 2015",WNDR assumes however all responsibility for th...,Kim LaCapria,A 14-year old schoolgirl has suffered serious ...,NaN,1,"Junk News, flu shot, world news daily report",67,1,false,0,true,0.997139,0.997139,0.000672,0.002190,[CLAIM] A teenage schoolgirl in Texas became p...


In [6]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# llm model from hw1
llm_model_id = "meta-llama/Llama-3.2-1B-Instruct"
llm_device = "cuda" if torch.cuda.is_available() else "cpu"

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    llm_model = llm_model.to(llm_device)

llm_model.eval()

llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id, use_fast=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [11]:
def extract_assistant_text(decoded: str) -> str:
    """
    Robustly extract only the assistant's answer from a full chat transcript.
    Handles cases where the model echoes system/user content.
    """
    s = decoded.strip()

    if "assistant" in s:
        s = s.split("assistant")[-1].strip()

    # Remove any lingering role headers that sometimes appear
    s = re.sub(r'^(system|user)\s*', '', s, flags=re.IGNORECASE).strip()

    return s

def to_two_sentences_str(text: str) -> str:
    text = text.strip()
    parts = re.split(r'(?<=[.!?])\s+', text)
    parts = [p.strip() for p in parts if p.strip()]
    return " ".join(parts[:2])

def run_llama_chat(llm_model, llm_tokenizer, messages, max_new_tokens=80):
    input_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = llm_tokenizer(input_text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(llm_model.device)
    attention_mask = inputs.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(llm_model.device)

    with torch.no_grad():
        out = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic (safer)
            temperature=0.0,
            top_p=1.0,
            repetition_penalty=1.05,
            pad_token_id=llm_tokenizer.eos_token_id,
            eos_token_id=llm_tokenizer.eos_token_id,
        )

    decoded = llm_tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(input_text):
        decoded = decoded[len(input_text):].strip()
    return decoded.strip()

def generate_explanation(prompt: str, max_new_tokens=80) -> str:
    messages = [
        {"role": "system", "content": "You write short, faithful classifier explanations."},
        {"role": "user", "content": prompt},
    ]
    raw = run_llama_chat(llm_model, llm_tokenizer, messages, max_new_tokens=max_new_tokens)
    assistant_only = extract_assistant_text(raw)
    return to_two_sentences_str(assistant_only)

def build_explanation_prompt(claim, predicted_label, confidence, evidence=None):
    evidence_block = ""
    if evidence and str(evidence).strip():
        evidence_block = f"\nEvidence (provided): {str(evidence).strip()}"

    return f"""You are generating a short, faithful explanation for a misinformation classifier.

Rules:
- Output EXACTLY 1–2 sentences.
- Do NOT add external facts.
- Only use the claim and the provided evidence (if any).
- Must be consistent with the predicted label.
- If label is 'mixture' or 'questionable', emphasize mixed/insufficient evidence and uncertainty.

Claim: {claim}
Predicted label: {predicted_label}
Model confidence: {confidence:.2f}{evidence_block}

Explanation:"""

In [12]:
bleu = BLEU()
bertscore = evaluate.load("bertscore")

def pick_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def generate_and_score_from_csv(
    df: pd.DataFrame,
    dataset_name: str,
    max_samples=200,
    claim_candidates=("claim","text","statement","headline"),
    evidence_candidates=("main_text","evidence","context","article","body"),
    gold_expl_candidates=("explanation","reasoning","rationale","human_explanation","justification","gold_explanation","label_explanation"),
    label_candidates=("pred_label","label","label_text"),
    conf_candidates=("pred_confidence","confidence","prob"),
    bert_model_type="distilbert-base-uncased",
):
    claim_col = pick_first_existing(df, claim_candidates)
    evidence_col = pick_first_existing(df, evidence_candidates)
    gold_col = pick_first_existing(df, gold_expl_candidates)
    label_col = pick_first_existing(df, label_candidates)
    conf_col = pick_first_existing(df, conf_candidates)

    print(f"\n[{dataset_name}] using claim={claim_col}, evidence={evidence_col}, gold={gold_col}, pred_label={label_col}, pred_conf={conf_col}")

    if claim_col is None or gold_col is None:
        raise ValueError(f"[{dataset_name}] Missing claim or gold explanation columns. Columns: {list(df.columns)}")

    if label_col is None:
        print(f"[{dataset_name}] WARNING: no predicted label column found; using 'mixture' for all.")
    if conf_col is None:
        print(f"[{dataset_name}] WARNING: no confidence column found; using 0.50 for all.")

    df_eval = df.head(max_samples).copy()

    preds, refs = [], []
    out_rows = []

    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"Generate ({dataset_name})"):
        claim = row.get(claim_col, "")
        gold = row.get(gold_col, "")

        if not isinstance(claim, str) or not claim.strip():
            continue
        if not isinstance(gold, str) or not gold.strip():
            continue

        evidence = row.get(evidence_col, None) if evidence_col else None
        pred_label = row.get(label_col, "mixture") if label_col else "mixture"

        conf = row.get(conf_col, 0.50) if conf_col else 0.50
        try:
            conf = float(conf)
        except:
            conf = 0.50

        prompt = build_explanation_prompt(claim, pred_label, conf, evidence=evidence)
        gen_expl = generate_explanation(prompt)

        preds.append(gen_expl)
        refs.append(gold)

        out_rows.append({
            "claim": claim,
            "gold_explanation": gold,
            "generated_explanation": gen_expl,
            "pred_label_used": pred_label,
            "pred_conf_used": conf,
        })

    if len(preds) == 0:
        raise ValueError(f"[{dataset_name}] No valid rows to score.")

    bleu_score = bleu.corpus_score(preds, [refs]).score

    bs = bertscore.compute(
        predictions=preds,
        references=refs,
        model_type=bert_model_type,
        lang="en",
        verbose=False
    )
    bert_p = float(np.mean(bs["precision"]))
    bert_r = float(np.mean(bs["recall"]))
    bert_f1 = float(np.mean(bs["f1"]))

    print(f"[{dataset_name}] BLEU: {bleu_score:.2f}")
    print(f"[{dataset_name}] BERTScore P/R/F1: {bert_p:.4f} / {bert_r:.4f} / {bert_f1:.4f}")

    return pd.DataFrame(out_rows), {
        "dataset": dataset_name,
        "n_scored": len(preds),
        "bleu": bleu_score,
        "bertscore_precision": bert_p,
        "bertscore_recall": bert_r,
        "bertscore_f1": bert_f1,
        "llm_model_id": llm_model_id,
        "bertscore_model_type": bert_model_type,
    }

intel_out, intel_metrics = generate_and_score_from_csv(
    intel_df,
    "intel_misclassified_top200",
    # intel: text + reasoning
    claim_candidates=("text","claim"),
    gold_expl_candidates=("reasoning","explanation"),
)

pub_out, pub_metrics = generate_and_score_from_csv(
    pub_df,
    "pubhealth_misclassified_top200",
    # pubhealth: claim + explanation + main_text
    claim_candidates=("claim","text"),
    gold_expl_candidates=("explanation","reasoning"),
    evidence_candidates=("main_text",),
)

metrics_df = pd.DataFrame([intel_metrics, pub_metrics])
display(metrics_df)


[intel_misclassified_top200] using claim=text, evidence=None, gold=reasoning, pred_label=pred_label, pred_conf=pred_confidence


Generate (intel_misclassified_top200):   0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[intel_misclassified_top200] BLEU: 9.64
[intel_misclassified_top200] BERTScore P/R/F1: 0.8176 / 0.8138 / 0.8151

[pubhealth_misclassified_top200] using claim=claim, evidence=main_text, gold=explanation, pred_label=pred_label, pred_conf=pred_confidence


Generate (pubhealth_misclassified_top200):   0%|          | 0/200 [00:00<?, ?it/s]

[pubhealth_misclassified_top200] BLEU: 0.65
[pubhealth_misclassified_top200] BERTScore P/R/F1: 0.8076 / 0.7449 / 0.7730


,dataset,n_scored,bleu,bertscore_precision,bertscore_recall,bertscore_f1,llm_model_id,bertscore_model_type
0,intel_misclassified_top200,200,9.638345,0.817559,0.813766,0.815140,meta-llama/Llama-3.2-1B-Instruct,distilbert-base-uncased
1,pubhealth_misclassified_top200,200,0.646564,0.807629,0.744882,0.772998,meta-llama/Llama-3.2-1B-Instruct,distilbert-base-uncased


In [13]:
# Save output
intel_save = "drive/My Drive/cs263_final_proj/intel_expl_llama32_1b_scored.csv"
pub_save = "drive/My Drive/cs263_final_proj/pubhealth_expl_llama32_1b_scored.csv"
metrics_save = "drive/My Drive/cs263_final_proj/expl_llama32_1b_metrics.csv"

intel_out.to_csv(intel_save, index=False)
pub_out.to_csv(pub_save, index=False)
metrics_df.to_csv(metrics_save, index=False)

print("Saved:")
print(intel_save)
print(pub_save)
print(metrics_save)

Saved:
drive/My Drive/cs263_final_proj/intel_expl_llama32_1b_scored.csv
drive/My Drive/cs263_final_proj/pubhealth_expl_llama32_1b_scored.csv
drive/My Drive/cs263_final_proj/expl_llama32_1b_metrics.csv
